# 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Kiểm tra kết nối
import os
GDRIVE_PATH = '/content/drive/MyDrive/DLPost'
print("GDrive connected:", os.path.exists(GDRIVE_PATH))
if os.path.exists(GDRIVE_PATH):
    print("Folders:", os.listdir(GDRIVE_PATH))
else:
    print("⚠️  Chưa có thư mục DLPost trên GDrive, sẽ tạo ở bước sau.")

# 2. Clone Git Repo


In [ ]:
import os

REPO_URL = "https://github.com/PNCanh/DLPost.git"
REPO_DIR = "/content/DLPost"

if os.path.exists(REPO_DIR):
    # Nếu đã clone rồi → pull code mới nhất
    %cd {REPO_DIR}
    !git pull origin main
    print("✅ Pulled latest code from GitHub")
else:
    # Clone lần đầu
    !git clone {REPO_URL} {REPO_DIR}
    print("✅ Cloned repo from GitHub")

%cd {REPO_DIR}
!git log --oneline -5  # Xem 5 commit gần nhất

# 3. Install Dependencies


In [ ]:
%cd /content/DLPost
!pip install -r requirements.txt -q
print("✅ Dependencies installed")

# 4. Path Config & Directory Setup


In [ ]:
import sys
import os

# Thêm src vào PYTHONPATH
sys.path.insert(0, '/content/DLPost/src')

# Set environment variable để paths.py detect Colab
# (COLAB_RELEASE_TAG đã có sẵn trên Colab, chỉ cần verify)
print(f"COLAB_RELEASE_TAG: {os.environ.get('COLAB_RELEASE_TAG', 'NOT SET')}")

# Import paths module (tự detect Colab)
from configs import paths
from configs.paths import ensure_directories

# Tạo tất cả thư mục cần thiết
ensure_directories()

# Verify paths
print(f"\n✅ Paths configured (IS_COLAB={paths.IS_COLAB}):")
print(f"   ROOT_DIR:       {paths.ROOT_DIR}")
print(f"   DATASET_DIR:    {paths.DATASET_DIR}")
print(f"   OUTPUT_DIR:     {paths.OUTPUT_DIR}")
print(f"   CHECKPOINTS_DIR:{paths.CHECKPOINTS_DIR}")
print(f"   LOGS_DIR:       {paths.LOGS_DIR}")
print(f"   PREDICTIONS_DIR:{paths.PREDICTIONS_DIR}")
print(f"   FIGURES_DIR:    {paths.FIGURES_DIR}")
print(f"   METRICS_DIR:    {paths.METRICS_DIR}")

# 5. Device Setup


In [ ]:
import torch

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU — using CPU")
    print("    → Vào Runtime > Change runtime type > GPU")

# 6. Select Config & Review

Chọn config để train từ `model_config.py`.

Mỗi config bao gồm: pooling strategy, fusion strategy, loss type, scheduler + warmup.

In [ ]:
# === Option A: Chạy full pipeline (OCR -> Preprocess -> Train) ===
# Chạy tất cả các steps
SELECTED_CONFIG = 'baseline'
!python /content/DLPost/src/main.py --config {SELECTED_CONFIG}

# === Option B: Chạy chỉ training (Bỏ qua OCR và Preprocess) ===
SELECTED_CONFIG = 'baseline'
!python /content/DLPost/src/main.py --config {SELECTED_CONFIG} --skip_ocr --skip_preprocess

In [ ]:
# === Option A: Chạy Full Pipeline (OCR -> Tiền xử lý & Chia dữ liệu -> Huấn luyện) ===
# Chạy toàn bộ luồng xử lý từ đầu nếu có dữ liệu mới hoặc có thay đổi cấu trúc
SELECTED_CONFIG = "baseline"  # Chọn cấu hình: baseline, variant_a, variant_b
!python /content/DLPost/src/main.py --config {SELECTED_CONFIG}

In [ ]:
# === Option B: Chỉ Huấn luyện (Bỏ qua OCR và Tiền xử lý) ===
# Sử dụng tập dữ liệu đã được OCR và tiền xử lý trước đó để huấn luyện mô hình
SELECTED_CONFIG = "baseline"  # Chọn cấu hình: baseline, variant_a, variant_b
!python /content/DLPost/src/main.py --config {SELECTED_CONFIG} --skip_ocr --skip_preprocess

# 8. Evaluate & Save Results


In [ ]:
from evaluators.evaluator import ClassificationEvaluator
from configs import paths
import json

# Tạo formatted predictions
formatted_preds = []
for i, item in test_df.iterrows():
    if i < len(predictions):
        pred_item = predictions[i]
        if isinstance(pred_item, dict) and "multiclass_probs" in pred_item:
            pred_label = int(np.argmax(pred_item["multiclass_probs"]))
        elif isinstance(pred_item, dict) and "binary_probs" in pred_item:
            pred_label = int(np.argmax(pred_item["binary_probs"]))
        else:
            pred_label = 0
            
        formatted_preds.append({
            "post_id": item.get("post_id", str(i)),
            "true_label": item.get("multi_label", 0),
            "predicted_label": pred_label,
            "explanation_probs": pred_item.get("explanation_probs", []) if isinstance(pred_item, dict) else [],
            "is_scam": pred_label != 0
        })

# Evaluate
evaluator = ClassificationEvaluator(
    run_name=config.get("run_name", "experiment"),
    model_name=model_name,
    timestamp=timestamp
)
mock_metrics = {"train_loss": [], "val_loss": []}
evaluator.evaluate(formatted_preds, mock_metrics)

print(f"\n✅ Results saved:")
print(f"   Checkpoints: {paths.CHECKPOINTS_DIR}")
print(f"   Logs:        {paths.LOGS_DIR}")
print(f"   Predictions: {paths.PREDICTIONS_DIR}")
print(f"   Figures:     {paths.FIGURES_DIR}")
print(f"   Metrics:     {paths.METRICS_DIR}")

# 9. Compare Models (Optional)

So sánh kết quả giữa các configs đã train.

In [ ]:
from evaluators.model_comparator import generate_comparison_report

# Tạo báo cáo so sánh giữa tất cả models đã train
generate_comparison_report()
print("✅ Comparison report generated.")